In [1]:
import pandas as pd
import yaml
import re

from pathlib import Path
from tqdm.auto import tqdm

from utils import count_ngrams
from utils import text_cleaning, normalize_sentence

%load_ext autoreload
%autoreload 2

## 1) Подсчёт вхождений токенов

In [ ]:
rrs = pd.read_parquet('data/rrs.parquet')
count_ngrams(rrs, 1)
count_ngrams(rrs, 2)
count_ngrams(rrs, 3)
count_ngrams(rrs, 4)

## 2) Выделение наиболее релевантных для кластеризации вакансий n-грамм. Делается вручную в Excel

## 3) Нормализация описаний вакансий

Делаем ту же обработку с помощью функции `text_cleaning`, что и при извлечении токенов Требований, Обязанностей и Условий. Единственное отличие - в той обработке некоторые теги сохранялись, теперь же мы будем удалять все теги.

In [2]:
TOKEN_EXTRACTION_CONFIG_PATH = Path.cwd().parent / 'configs' / 'token_extraction.yaml'
with TOKEN_EXTRACTION_CONFIG_PATH.open(encoding='utf-8') as f:
    extraction_config = yaml.safe_load(f)

NECESSARY_PUNCTUATION = extraction_config.get("NECESSARY_PUNCTUATION", [])
PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES = extraction_config.get("PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES", [])

PLUSES = extraction_config.get("PLUSES", [])
MINUSES = extraction_config.get("MINUSES", [])
DOTS = extraction_config.get("DOTS", [])

In [3]:
cleaned_vacancies = pd.read_csv("../headhunter_parsing/data/vacancies_cleaned.csv")

In [4]:
punctuation = set()
for text in cleaned_vacancies.description:
    new_symbols = set(re.findall(r'[^\wё]', text.lower()))
    punctuation |= new_symbols
RUBBISH_PUNCTUATION = punctuation - set(NECESSARY_PUNCTUATION)

In [5]:
rows = []
columns = list(cleaned_vacancies.columns)

for tuple in tqdm(cleaned_vacancies.itertuples(index = False, name = None)):
    row = dict(zip(columns, tuple))
    id = row.get("id", "")
    description = row.get("description", "")

    cleaned_description = text_cleaning(
        description, PLUSES, DOTS, MINUSES, RUBBISH_PUNCTUATION, 
        PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES, [], [], delete_all_tags=True
        )
    normalized_description = normalize_sentence(cleaned_description)
    rows.append([id, normalized_description])

resulting_table = pd.DataFrame(rows, columns=['id', 'normalized_description'])
resulting_table.to_parquet('data/normalized_descriptions.parquet', index = False)

0it [00:00, ?it/s]